[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/thermodynamics/phase_change/phase_change.ipynb)

# Boiling at Fixed Pressure

This example computes how a van der Waals fluid boils: heat taken in at fixed pressure warms the liquid, then turns it into vapour at a fixed temperature, then warms the vapour. Below the critical point, liquid and vapour coexist at the two volumes where their pressures and their chemical potentials are equal, and the heat that boiling takes is the jump in enthalpy between them, the latent heat.

A state of coexistence is modelled as the two phases' volumes, an antivector of the plane spanned by the liquid's volume `mv.l.dual()` and the vapour's `mv.g.dual()`, each read back by pairing, `mv.l & state`. How far a state is from coexisting is a function from states to states, and its derivative at a state is a map from a correction of the state to the change it makes; Newton's method solves that map for the correction that cancels the mismatch. The solve runs over a batch of temperatures as over one, and so draws the whole dome of coexisting volumes at once.

In the usual notation, coexistence is Maxwell's construction: $p(V_l) = p(V_g)$ and $\mu(V_l) = \mu(V_g)$ with $\mu = F + pV$, which cuts the isotherm's loop at the saturation pressure into two equal areas, $\int_{V_l}^{V_g} p\,dV = p_\text{sat}(V_g - V_l)$. The heat taken in at fixed pressure is $dQ = dH$, and the latent heat is $L = H_g - H_l = T(S_g - S_l)$.

Temperature and volume are in their critical values, energy in $RT_c$ and pressure in $RT_c/V_c$, for a fixed amount of the fluid.

In [ ]:
# In Colab, numga and its examples, from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples,jax] @ git+https://github.com/EelcoHoogendoorn/numga"

In [ ]:
%matplotlib inline
import jax
import numpy as np

from numga import Algebra
from numga.backend.jax import JaxContext, derivative
from examples.util.animation import show_animation
from examples.thermodynamics.phase_change import render

jax.config.update("jax_enable_x64", True)
ga = Algebra("l0g0")
mv = JaxContext(ga, np.float64).multivector
Scalar = ga.gatype.scalar()
State = ga.gatype.antivector()

# The fluid: its heat capacity at fixed volume, the strength of its molecules' attraction and the volume they exclude,
# in the critical units, where the critical point lies at unit temperature, unit volume and pressure 3/8.
capacity, attraction, exclusion = 1.5, 9 / 8, 1 / 3

## Liquid and vapour in coexistence

The fluid's pressure falls as its volume grows, except along a loop below the critical temperature where a uniform fluid would be unstable and splits into liquid and vapour instead. The two phases coexist at the volumes where they have the same pressure and the same chemical potential, `helmholtz_energy + pressure * volume`. Their mismatch is the state `mv.l.dual() * pressure_gap + mv.g.dual() * chemical_gap`, and each Newton step subtracts `derivative(gap)(state).solve(gap(state))`. The step runs batched over the temperatures the charts below are drawn at, and over the temperature the fluid will boil at.

In [ ]:
def pressure(volume: Scalar, temperature: np.ndarray) -> Scalar:
    """The van der Waals equation of state."""
    return temperature / (volume - exclusion) - attraction / volume.squared()

def helmholtz_energy(volume: Scalar, temperature: np.ndarray) -> Scalar:
    """The Helmholtz energy, whose slope along the volume is minus the pressure."""
    return -capacity * temperature * np.log(temperature) - temperature * (volume - exclusion).log() - attraction / volume

def chemical_potential(volume: Scalar, temperature: np.ndarray) -> Scalar:
    """Where the Helmholtz energy's tangent at a volume crosses zero volume."""
    return helmholtz_energy(volume, temperature) + pressure(volume, temperature) * volume

def mismatch(state: State, temperature: np.ndarray) -> State:
    """The difference in pressure between the state's two volumes, along the liquid's, and in chemical potential,
    along the vapour's."""
    liquid, vapour = mv.l & state, mv.g & state
    pressure_gap = pressure(liquid, temperature) - pressure(vapour, temperature)
    chemical_gap = chemical_potential(liquid, temperature) - chemical_potential(vapour, temperature)
    return mv.l.dual() * pressure_gap + mv.g.dual() * chemical_gap

def coexistence(temperature: np.ndarray) -> State:
    """The coexisting volumes at each temperature `[...]`, by Newton's method."""
    # Below the critical point the two volumes part as the square root of the distance from it, the liquid's
    # slowly, the vapour's fast; this starts each on its own side.
    below = np.sqrt(1 - temperature)
    state = mv.l.dual() / (1 + 2 * below) + mv.g.dual() / (1 - below) ** 3     # [...] State
    gap = lambda state: mismatch(state, temperature)
    for _ in range(n_newton):
        state = state - derivative(gap)(state).solve(gap(state))               # [...] State
    return state

# The charts' second reading: the enthalpy, the heat taken in at fixed pressure being its change, and the equation of
# state solved for the temperature, which traces an isobar.
def enthalpy(volume: Scalar, temperature: np.ndarray) -> Scalar:
    return capacity * temperature - attraction / volume + pressure(volume, temperature) * volume

def isobar_temperature(volume: Scalar, level: float) -> Scalar:
    return (level + attraction / volume.squared()) * (volume - exclusion)

n_newton = 8
boiling_temperature = 0.85
render.draw_charts(pressure, enthalpy, coexistence, isobar_temperature, boiling_temperature)

The pressure–volume chart on the left holds isotherms, the critical one darker, under the dome of coexisting volumes in green: the liquid's volumes up its left side to the critical point and the vapour's down its right. The dark isotherm at the boiling temperature crosses its loop at the saturation pressure, which cuts off two equal areas, blue below and orange above. The temperature–enthalpy chart on the right holds the dome again and the isobars, each rising through the liquid, flat across the dome while the fluid boils and rising through the vapour; the dashed lines across the dome are those of equal vapour fraction.

## Heating at fixed pressure

A weight on the piston holds the pressure at the saturation pressure of the boiling temperature, and heat comes in at a steady rate. The heat taken in at fixed pressure is the enthalpy's change, so the fluid walks along its isobar in equal steps of enthalpy. The liquid warms to the boiling temperature, where it turns into vapour without warming, its volume and enthalpy shared out between the two phases by the vapour's fraction; then the vapour warms. Where the isobar reaches a temperature, Newton's method finds the volume from the equation of state solved for the temperature.

This cell takes about half a minute, most of it drawing the animation.

In [ ]:
def volume_at(temperature: np.ndarray, level: float, guess: Scalar) -> Scalar:
    """The volume where the isobar at a pressure level reaches each temperature, from a guess on the liquid's or the
    vapour's side."""
    gap = lambda volume: isobar_temperature(volume, level) - temperature
    volume = guess
    for _ in range(n_newton):
        volume = volume - derivative(gap)(volume).solve(gap(volume))
    return volume

def mixture(coexisting: State, vapour_fraction: float, temperature: float) -> tuple[Scalar, Scalar, Scalar]:
    """The volume and the enthalpy of a mixture with a fraction of its amount as vapour, each shared out between the
    phases by that fraction, and the volume its liquid takes up."""
    liquid, vapour = mv.l & coexisting, mv.g & coexisting
    volume = (1 - vapour_fraction) * liquid + vapour_fraction * vapour
    shared = (1 - vapour_fraction) * enthalpy(liquid, temperature) + vapour_fraction * enthalpy(vapour, temperature)
    return volume, shared, (1 - vapour_fraction) * liquid

coldest, hottest, n_frames = 0.7, 0.98, 60
coexisting = coexistence(np.array(boiling_temperature))
liquid, vapour = mv.l & coexisting, mv.g & coexisting
level = float(pressure(liquid, boiling_temperature).to_array())

def heating():
    """The fluid's volume, temperature, enthalpy and the volume its liquid takes up, in equal steps of heat."""
    # The enthalpies where the stages start and end: each stage takes steps in proportion to its share of the heat.
    ends = [enthalpy(volume_at(coldest, level, liquid), coldest), enthalpy(liquid, boiling_temperature),
            enthalpy(vapour, boiling_temperature), enthalpy(volume_at(hottest, level, vapour), hottest)]
    spans = np.diff([float(end.to_array()) for end in ends])
    counts = np.round(n_frames * spans / spans.sum()).astype(int)
    for temperature in np.linspace(coldest, boiling_temperature, counts[0], endpoint=False):
        volume = volume_at(temperature, level, liquid)
        yield volume, mv.scalar([temperature]), enthalpy(volume, temperature), volume
    for vapour_fraction in np.linspace(0, 1, counts[1], endpoint=False):
        volume, shared, liquid_volume = mixture(coexisting, vapour_fraction, boiling_temperature)
        yield volume, mv.scalar([boiling_temperature]), shared, liquid_volume
    for temperature in np.linspace(boiling_temperature, hottest, counts[2]):
        volume = volume_at(temperature, level, vapour)
        yield volume, mv.scalar([temperature]), enthalpy(volume, temperature), volume * 0

show_animation(render.heating(pressure, enthalpy, coexistence, isobar_temperature, boiling_temperature, heating()),
               "phase_change", duration_ms=70)

The liquid at the bottom of the cylinder warms with hardly a change of volume; at the boiling temperature the vapour above it grows and lifts the loaded piston while the temperature holds, the state running along the flat stretch on the left and the flat stretch of the dark isobar on the right; once the liquid is gone the vapour warms and expands. The flat stretch on the right is as long as the latent heat, and the steady heating spends most of its time there.

In [ ]:
latent_heat = enthalpy(vapour, boiling_temperature) - enthalpy(liquid, boiling_temperature)   # [] Scalar

In [ ]:
print("coexisting volumes:", (mv.l & coexisting).to_array(), (mv.g & coexisting).to_array())
print("saturation pressure:", level)
print("latent heat:", latent_heat.to_array())

In [ ]:
# checks
np.testing.assert_allclose(mismatch(coexisting, boiling_temperature).kernel, 0, atol=1e-10)
# The latent heat is the temperature times the jump in entropy, the logarithm of the free volumes' ratio.
np.testing.assert_allclose(latent_heat.kernel, (boiling_temperature * ((vapour - exclusion) / (liquid - exclusion)).log()).kernel, atol=1e-10)
# The isobar's volume at a temperature has the isobar's pressure.
np.testing.assert_allclose(pressure(volume_at(coldest, level, liquid), coldest).kernel, level, atol=1e-10)
# Clapeyron: the saturation pressure rises with the temperature as the latent heat over the temperature and the jump in volume.
step = 1e-4
rise = (pressure(mv.l & coexistence(np.array(boiling_temperature + step)), boiling_temperature + step)
        - pressure(mv.l & coexistence(np.array(boiling_temperature - step)), boiling_temperature - step)) / (2 * step)
np.testing.assert_allclose(rise.kernel, (latent_heat / (boiling_temperature * (vapour - liquid))).kernel, rtol=1e-6)